# Оптимізація розташування центру обслуговування на 3D‑рельєфі


![Main photo](main_photo.png)


## Мета
Розробити математичну модель та програмний алгоритм для знаходження оптимального місця розташування центру обслуговування на тривимірному рельєфі. Оптимальною вважається точка, яка мінімізує сумарні витрати на обслуговування (або сумарну відстань) до заданої множини клієнтів.

## Вступ
Певно, що в житті, стикаючись із задачами на оптимізацію, люди ніколи не працюють на ідеальній площині. Наявність висот, пагорбів та інших перешкод суттєво впливає на відстані та вартість переміщення. __Задача Вебера__ шукає точку на площині, сума відстаней від якої до заданих точок є мінімальною. У нашій роботі ми розширюємо цю задачу: наш простір є рандомно згенерованою тривимірною поверхнею (рельєфом), а пошук оптимальної локації вимагає врахування перепадів висот. Це має пряме застосування у проєктуванні доріг, розміщенні станцій зв'язку, логістичних хабів - одним словом дуже важливий проєкт! :)

## Постановка задачі
Нехай задано ділянку місцевості, рельєф якої описується функцією висоти $z=f(x, y)$. 
На цій поверхні розташовано $N$ клієнтів у точках $P_i=(x_i, y_i, z_i)$, де $z_i=f(x_i, y_i)$ для $i=1, 2, ..., N$.

Необхідно знайти координати центру обслуговування $C=(x^*, y^*, z^*)$, де $z^*=f(x^*, y^*)$, що мінімізує цільову функцію сумарних витрат (відстаней):

$$W(x, y)=\sum_{i=1}^{N} w_i \cdot d(C, P_i)$$

де:
* $w_i$ — вага (важливість або обсяг попиту) $i$-го клієнта;
  Для простого обчислення ваги ми вирішили що, логічно, неоптимальним є шлях, який іде під кутом, отже нашим ваговим коефіцієнтом є $ w_i = 1 + |tan(\alpha)| $
* $d(C, P_i)$ — евклідова відстань у 3D-просторі між центром та клієнтом:

$$d(C, P_i)=\sqrt{(x^* - x_i)^2 + (y^* - y_i)^2 + (f(x^*, y^*) - z_i)^2}$$

## План роботи
1. **Генерація та інтерактивна візуалізація середовища:** Створення випадкового згладженого 3D-рельєфу (суперпозиція функцій Гауса) та випадкове розміщення клієнтів. Побудова інтерактивної моделі, яку можна обертати для аналізу ландшафту.
2. **Побудова математичної моделі:** Програмування цільової функції (функції витрат), яка обчислює сумарну евклідову відстань від будь-якої точки на поверхні до всіх клієнтів.
3. **Аналіз поля витрат (Heatmap):** Візуалізація "поверхні вартості" поверх рельєфу. Це дозволить наочно побачити зони, де будівництво центру буде найдешевшим.
4. **Оптимізація:** Застосування чисельних методів для пошуку глобального мінімуму функції витрат (точки ідеального розташування).
5. **Аналіз результатів:** Фінальне відображення знайденого оптимального центру на рельєфі та формування висновків.

### 1. Генерація 3D-середовища та клієнтів

Основна фішка нашого проєкту — **динамічність**. Ми не використовуємо статичні карти. Щоразу генерується абсолютно новий ландшафт, під який автоматично адаптуються всі подальші розрахунки.

Щоб рельєф виглядав природно, ми використовуємо суперпозицію двовимірних функцій Гауса. По суті, ми випадково розкидаємо по площині "пагорби", і висота $z$ у будь-якій точці $(x, y)$ обчислюється як сума їхніх впливів:

$$z(x, y) = \sum_{i=1}^{m} h_i \cdot \exp\left(-\frac{(x - x_{0i})^2 + (y - y_{0i})^2}{2\sigma_i^2}\right)$$

Де $h_i$ — це висота $i$-го пагорба, $(x_{0i}, y_{0i})$ — координати його вершини, а $\sigma_i$ — похилість схилу. Усі ці параметри зберігаються в масиві `HILLS_DATA` для подальшого використання.

Після формування рельєфу ми відмічаємо на ньому $N$ клієнтів. Їхні координати $(x, y)$ обираються випадково, а висота $z$ вираховується через нашу функцію рельєфу — так вони всі ідеально належать графіку. Їхні точні позиції записуються в `CLIENTS_DATA`, і ми готові до оптимізації!

In [3]:
var('x, y')
import random
from sage.plot.colors import colormaps

# Зберігаємо параметри рельєфу (координати центру, висота, ширина пагорба)
num_hills = 30
HILLS_DATA = []
for _ in range(num_hills):
    x0 = random.uniform(-20, 20)
    y0 = random.uniform(-20, 20)
    h = random.uniform(0.8, 6) # амплітуда
    sigma = random.uniform(3.0, 6.0)
    HILLS_DATA.append((x0, y0, h, sigma))

# Математична функція поверхні (буде потрібна і для оптимізації)
def terrain_func(x_val, y_val):
    z = 0
    for x0, y0, h, sigma in HILLS_DATA:
        d2 = (x_val - x0)**2 + (y_val - y0)**2
        z += h * exp(-d2 / (2 * sigma**2))
    return z

# Зберігаємо координати клієнтів (x, y, z)
N = 20
CLIENTS_DATA = []
for _ in range(N):
    cx = random.uniform(-20, 20)
    cy = random.uniform(-20, 20)
    cz = terrain_func(cx, cy)
    CLIENTS_DATA.append((cx, cy, cz))


# Розрахунок кольору для рельєфу
z_vals = [terrain_func(i, j) for i in range(-20, 21, 2) for j in range(-20, 21, 2)]
z_min, z_max = min(z_vals), max(z_vals)
color_func = lambda x, y: float((terrain_func(x, y) - z_min) / (z_max - z_min + 0.001))

# Малюємо рельєф
terrain_plot = plot3d(terrain_func, (x, -20, 20), (y, -20, 20), 
                      color=(color_func, colormaps.terrain), 
                      plot_points=100, 
                      opacity=0.9)

# Малюємо клієнтів як об'ємні сфери (радіус 0.6)
# Ми додаємо +0.3 до координати Z, щоб сфера "сиділа" на землі, а не ховалася в ній
clients_plot = sum([sphere((cx, cy, cz + 0.3), size=0.6, color='red') for cx, cy, cz in CLIENTS_DATA])

title_text = text3d("Згенерований 3D-рельєф із клієнтами", (0, 0, z_max + 4), color='black', fontsize=14)
label_x = text3d("Вісь X", (22, 0, 0), color='black')
label_y = text3d("Вісь Y", (0, 22, 0), color='black')

(terrain_plot + clients_plot + title_text + label_x + label_y).show(frame=True)

Graphics3d Object